# Lab #05: 인공 신경망 (Artificial Neural Network)

**딥러닝 응용 (Deep Learning Application)**

국립금오공과대학교 산업·빅데이터공학부 | 추상현

---

## 목차

| 파트 | 주제 | 내용 |
|:---:|:---|:---|
| **Part 1** | 단층 퍼셉트론 (SLP) | 퍼셉트론, 논리 게이트, XOR 한계 |
| **Part 2** | 다층 퍼셉트론 (MLP) | MLP 구조, XOR 해결, 활성화 함수 |
| **Part 3** | 역전파 (Backpropagation) | 연쇄법칙, BP 알고리즘, 학습 루프 |
| **Part 4** | 종합 실습 | MNIST 손글씨 분류 프로젝트 |

---

## 🎯 학습 목표 (Learning Objectives)

- 단층 퍼셉트론(SLP)의 구조와 원리 이해
- 다층 퍼셉트론(MLP)의 구조와 비선형 문제 해결 원리 이해
- 역전파(Backpropagation) 알고리즘의 수학적 원리 이해 및 구현
- PyTorch를 활용한 신경망 구축 및 학습

---

## ⚙️ 환경 설정 (Environment Setup)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Device 설정
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🖥️ Using GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print("🖥️ Using Apple MPS")
else:
    device = torch.device('cpu')
    print("🖥️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

---
# Part 1: 단층 퍼셉트론 (Single-Layer Perceptron)

## 1.1 생물학적 뉴런에서 인공 뉴런으로

생물학적 뉴런의 구조를 인공 뉴런(퍼셉트론)에 대응시킬 수 있습니다.

| 생물학적 뉴런 | 인공 뉴런 (Perceptron) |
|:---:|:---:|
| 수상돌기 (Dendrites) | 입력 (Inputs, $x_i$) |
| 시냅스 강도 (Synapse Strength) | 가중치 (Weights, $w_i$) |
| 세포체 (Cell Body) | 가중합 + 활성화 (Weighted Sum + Activation) |
| 축삭돌기 (Axon) | 출력 (Output, $y$) |

### 퍼셉트론의 수학적 표현

1. **가중합 (Weighted Sum):**

$$z = \sum_{i=1}^{n} w_i x_i + b = \mathbf{w}^T \mathbf{x} + b$$

2. **활성화 (Activation):**

$$y = f(z)$$

3. **계단 함수 (Step Function):**

$$f(z) = \begin{cases} 1 & \text{if } z \geq 0 \\ 0 & \text{if } z < 0 \end{cases}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def perceptron(x, w, b):
    """단일 퍼셉트론"""
    z = np.dot(w, x) + b  # Weighted sum
    return 1 if z >= 0 else 0  # Step function

# AND gate example
w = np.array([1.0, 1.0])
b = -1.5

print("=== AND Gate (Perceptron) ===")
for x1 in [0, 1]:
    for x2 in [0, 1]:
        output = perceptron(np.array([x1, x2]), w, b)
        print(f"  Input: ({x1}, {x2}) → Output: {output}")

# Visualize decision boundary
fig, ax = plt.subplots(figsize=(6, 6))
# Plot data points
for x1 in [0, 1]:
    for x2 in [0, 1]:
        output = perceptron(np.array([x1, x2]), w, b)
        color = 'red' if output == 1 else 'blue'
        ax.scatter(x1, x2, c=color, s=200, zorder=5, edgecolors='k')
        ax.annotate(f'{output}', (x1+0.05, x2+0.05), fontsize=14, fontweight='bold')

# Decision boundary: w1*x1 + w2*x2 + b = 0
x1_line = np.linspace(-0.5, 1.5, 100)
x2_line = -(w[0] * x1_line + b) / w[1]
ax.plot(x1_line, x2_line, 'g--', linewidth=2, label='Decision Boundary')
ax.fill_between(x1_line, x2_line, 2, alpha=0.1, color='red')
ax.fill_between(x1_line, x2_line, -1, alpha=0.1, color='blue')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('x1', fontsize=12); ax.set_ylabel('x2', fontsize=12)
ax.set_title('AND Gate - Perceptron Decision Boundary', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 1.2 SLP로 논리 게이트 구현 (AND, OR)

### 진리표 (Truth Table)

| $x_1$ | $x_2$ | AND | OR |
|:---:|:---:|:---:|:---:|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 |
| 1 | 0 | 0 | 1 |
| 1 | 1 | 1 | 1 |

### 가중치 설정

| Gate | $w_1$ | $w_2$ | $b$ |
|:---:|:---:|:---:|:---:|
| AND | 1.0 | 1.0 | -1.5 |
| OR | 1.0 | 1.0 | -0.5 |

In [ ]:
# AND와 OR 게이트 비교
gates = {
    'AND': {'w': np.array([1.0, 1.0]), 'b': -1.5},
    'OR':  {'w': np.array([1.0, 1.0]), 'b': -0.5}
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (gate_name, params) in enumerate(gates.items()):
    ax = axes[idx]
    w, b = params['w'], params['b']

    print(f"=== {gate_name} Gate ===")
    for x1 in [0, 1]:
        for x2 in [0, 1]:
            output = perceptron(np.array([x1, x2]), w, b)
            color = 'red' if output == 1 else 'blue'
            ax.scatter(x1, x2, c=color, s=200, zorder=5, edgecolors='k')
            ax.annotate(f'{output}', (x1+0.05, x2+0.05), fontsize=14, fontweight='bold')
            print(f"  ({x1}, {x2}) → {output}")

    # Decision boundary
    x1_line = np.linspace(-0.5, 1.5, 100)
    x2_line = -(w[0] * x1_line + b) / w[1]
    ax.plot(x1_line, x2_line, 'g--', linewidth=2, label='Decision Boundary')
    ax.fill_between(x1_line, x2_line, 2, alpha=0.1, color='red')
    ax.fill_between(x1_line, x2_line, -1, alpha=0.1, color='blue')
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x1', fontsize=12); ax.set_ylabel('x2', fontsize=12)
    ax.set_title(f'{gate_name} Gate - Decision Boundary', fontsize=14, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    print()

plt.tight_layout(); plt.show()

### 🔗 AI 연결: 선형 분리 가능성 (Linear Separability)

> SLP는 **선형 결정 경계**(직선)만 그릴 수 있습니다. 데이터가 직선 하나로 분리 가능하면 SLP로 풀 수 있지만, 그렇지 않은 경우(예: XOR)에는 SLP로 해결할 수 없습니다.

### 📝 실습 1-1 [초급] NAND 게이트 구현

NAND 게이트의 진리표:

| $x_1$ | $x_2$ | NAND |
|:---:|:---:|:---:|
| 0 | 0 | 1 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

**미션:** NAND 게이트를 구현하는 가중치($w_1, w_2$)와 바이어스($b$)를 찾으세요.

In [ ]:
# 📝 실습 1-1: NAND 게이트 구현

# NAND 가중치와 바이어스를 설정하세요
w_nand = ____  # np.array([?, ?])
b_nand = ____  # 바이어스 값

# 테스트
print("=== NAND Gate Test ===")
expected = [1, 1, 1, 0]
results = []
for i, (x1, x2) in enumerate([(0,0), (0,1), (1,0), (1,1)]):
    output = perceptron(np.array([x1, x2]), w_nand, b_nand)
    results.append(output)
    status = "✓" if output == expected[i] else "✗"
    print(f"  Input: ({x1}, {x2}) → Output: {output} (Expected: {expected[i]}) {status}")

# 검증
assert results[0] == 1, f"NAND(0,0) should be 1, got {results[0]}"
assert results[1] == 1, f"NAND(0,1) should be 1, got {results[1]}"
assert results[2] == 1, f"NAND(1,0) should be 1, got {results[2]}"
assert results[3] == 0, f"NAND(1,1) should be 0, got {results[3]}"

print("\n✅ 모든 테스트 통과!")

## 1.3 SLP = Linear Regression / Logistic Regression

SLP의 활성화 함수를 바꾸면 우리가 이미 배운 모델이 됩니다!

| SLP 구성요소 | 활성화 함수: Identity ($f(z)=z$) | 활성화 함수: Sigmoid ($\sigma(z)$) |
|:---:|:---:|:---:|
| **모델 이름** | Linear Regression | Logistic Regression |
| **출력 범위** | $(-\infty, +\infty)$ | $(0, 1)$ |
| **용도** | 회귀 (Regression) | 이진 분류 (Binary Classification) |
| **손실 함수** | MSE Loss | BCE Loss |

즉, **SLP는 가장 단순한 신경망**이며, 우리가 이미 배운 선형/로지스틱 회귀와 동일합니다!

In [ ]:
# PyTorch로 SLP (1-neuron) 구현 — 간단한 분류 문제
from sklearn.datasets import make_blobs

# 데이터 생성
np.random.seed(42)
X_np, y_np = make_blobs(n_samples=200, centers=2, n_features=2, random_state=42, cluster_std=1.5)
X_data = torch.tensor(X_np, dtype=torch.float32)
y_data = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

# SLP 모델 (= Logistic Regression)
slp_model = nn.Sequential(
    nn.Linear(2, 1),
    nn.Sigmoid()
)

criterion_slp = nn.BCELoss()
optimizer_slp = torch.optim.SGD(slp_model.parameters(), lr=0.01)

# 학습
for epoch in range(200):
    output = slp_model(X_data)
    loss = criterion_slp(output, y_data)
    optimizer_slp.zero_grad()
    loss.backward()
    optimizer_slp.step()

# 결과
with torch.no_grad():
    preds = (slp_model(X_data) > 0.5).float()
    acc = (preds == y_data).float().mean()
    print(f"SLP (Logistic Regression) Accuracy: {acc.item():.4f}")

# 시각화
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_np[y_np==0, 0], X_np[y_np==0, 1], c='blue', label='Class 0', alpha=0.6)
ax.scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='red', label='Class 1', alpha=0.6)

# Decision boundary
with torch.no_grad():
    w = slp_model[0].weight.numpy().flatten()
    b_val = slp_model[0].bias.numpy().item()
    x_min, x_max = X_np[:, 0].min() - 1, X_np[:, 0].max() + 1
    x_plot = np.linspace(x_min, x_max, 100)
    y_plot = -(w[0] * x_plot + b_val) / w[1]
    ax.plot(x_plot, y_plot, 'g--', linewidth=2, label='Decision Boundary')

ax.set_xlabel('Feature 1', fontsize=12); ax.set_ylabel('Feature 2', fontsize=12)
ax.set_title('SLP Binary Classification', fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### 📝 실습 1-2 [초급] SLP로 간단한 이진 분류

`make_blobs`로 생성한 2D 데이터를 단일 뉴런(`nn.Linear(2,1)` + `sigmoid`)으로 분류하세요.

In [ ]:
# 📝 실습 1-2: SLP 이진 분류

from sklearn.datasets import make_blobs

# 데이터 생성
X_ex, y_ex = make_blobs(n_samples=300, centers=2, n_features=2, random_state=0, cluster_std=1.0)
X_train = torch.tensor(X_ex, dtype=torch.float32)
y_train = torch.tensor(y_ex, dtype=torch.float32).unsqueeze(1)

# 모델 정의
my_slp = ____  # nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())

# 손실함수와 옵티마이저
bce_loss = nn.BCELoss()
opt = torch.optim.SGD(my_slp.parameters(), lr=0.01)

# 학습
for epoch in range(300):
    outputs = ____  # 모델에 X_train을 입력
    loss = ____  # BCELoss 계산
    opt.zero_grad()
    loss.backward()
    opt.step()

# 평가
with torch.no_grad():
    final_preds = (my_slp(X_train) > 0.5).float()
    accuracy = (final_preds == y_train).float().mean().item()
    print(f"SLP Accuracy: {accuracy:.4f}")

assert accuracy > 0.9, f"Accuracy {accuracy:.4f} < 0.9. 모델을 확인하세요!"
print("\n✅ 모든 테스트 통과!")

## 1.4 XOR 문제 — SLP의 한계

### XOR 진리표

| $x_1$ | $x_2$ | XOR |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

XOR 데이터는 **직선 하나로 분리할 수 없습니다!** 이것이 바로 SLP의 한계입니다.

In [ ]:
# XOR 데이터 시각화
xor_data = np.array([[0,0],[0,1],[1,0],[1,1]])
xor_labels = np.array([0, 1, 1, 0])

fig, ax = plt.subplots(figsize=(6, 6))
for i in range(4):
    color = 'red' if xor_labels[i] == 1 else 'blue'
    ax.scatter(xor_data[i,0], xor_data[i,1], c=color, s=300, zorder=5, edgecolors='k', linewidth=2)
    ax.annotate(f'XOR={xor_labels[i]}', (xor_data[i,0]+0.05, xor_data[i,1]+0.08), fontsize=12, fontweight='bold')

ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('x1', fontsize=12); ax.set_ylabel('x2', fontsize=12)
ax.set_title('XOR Problem - Not Linearly Separable!', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# SLP로 XOR 학습 시도 → 실패!
X_xor = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y_xor = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

slp_xor = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
criterion_xor = nn.BCELoss()
optimizer_xor = torch.optim.SGD(slp_xor.parameters(), lr=0.5)

losses_xor = []
for epoch in range(1000):
    out = slp_xor(X_xor)
    loss = criterion_xor(out, y_xor)
    optimizer_xor.zero_grad()
    loss.backward()
    optimizer_xor.step()
    losses_xor.append(loss.item())

with torch.no_grad():
    preds_xor = (slp_xor(X_xor) > 0.5).float()
    acc_xor = (preds_xor == y_xor).float().mean()
    print(f"SLP on XOR - Accuracy: {acc_xor.item():.4f}")
    print("XOR predictions:", preds_xor.flatten().tolist())

plt.figure(figsize=(8, 4))
plt.plot(losses_xor)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('SLP on XOR - Training Loss (Does Not Converge!)', fontweight='bold')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### 📝 실습 1-3 [중급] XOR에 SLP 적용 → 실패 원인 분석

위 코드를 실행한 후, 결정 경계를 시각화하고 실패 원인을 분석하세요.

In [ ]:
# 📝 실습 1-3: XOR SLP 결정 경계 시각화

# SLP의 결정 경계 시각화
fig, ax = plt.subplots(figsize=(6, 6))

# 배경 색상 (결정 영역)
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    Z = slp_xor(grid).numpy().reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.6)
ax.contour(xx, yy, Z, levels=[0.5], colors='green', linewidths=2, linestyles='--')

# 데이터 포인트
for i in range(4):
    color = 'red' if xor_labels[i] == 1 else 'blue'
    ax.scatter(xor_data[i,0], xor_data[i,1], c=color, s=300, zorder=5, edgecolors='k', linewidth=2)

ax.set_xlabel('x1', fontsize=12); ax.set_ylabel('x2', fontsize=12)
ax.set_title('SLP Decision Boundary on XOR (Fails!)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("\nSLP Accuracy on XOR:", acc_xor.item())
print("→ SLP는 직선(선형 결정 경계) 하나만 그릴 수 있으므로 XOR을 풀 수 없습니다!")

**질문:** 왜 SLP는 XOR을 풀 수 없는가?

> **답변을 아래에 작성하세요:**
>
> (힌트: 선형 분리 가능성, 결정 경계의 형태를 생각해보세요)

---
# Part 2: 다층 퍼셉트론 (Multi-Layer Perceptron)

## 2.1 MLP 구조

MLP는 **입력층(Input Layer)**, **은닉층(Hidden Layer)**, **출력층(Output Layer)**으로 구성됩니다.

### 순전파 (Forward Pass) 수식

**은닉층:**
$$\mathbf{h} = f(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1)$$

**출력층:**
$$\mathbf{y} = g(\mathbf{W}_2 \mathbf{h} + \mathbf{b}_2)$$

### 주요 용어

| 용어 | 설명 |
|:---:|:---|
| Weights ($\mathbf{W}$) | 뉴런 간 연결 강도 |
| Biases ($\mathbf{b}$) | 각 뉴런의 편향 |
| Neurons | 각 층의 노드 |
| Pre-activations ($\mathbf{z}$) | 활성화 함수 적용 전 값 |
| Activations ($\mathbf{h}$) | 활성화 함수 적용 후 값 |

### 파라미터 수 계산 예시

입력 2개, 은닉 3개, 출력 1개인 MLP:
- 1층: $(2 \times 3) + 3 = 9$ (가중치 + 바이어스)
- 2층: $(3 \times 1) + 1 = 4$
- **총 파라미터: 13개**

In [ ]:
# MLP 구조 시각화
fig, ax = plt.subplots(figsize=(10, 6))

layer_sizes = [2, 3, 1]
layer_names = ['Input\nLayer', 'Hidden\nLayer', 'Output\nLayer']
colors = ['#3498db', '#e74c3c', '#2ecc71']

positions = {}
for l, size in enumerate(layer_sizes):
    x = l * 3
    for n in range(size):
        y = (size - 1) / 2 - n
        positions[(l, n)] = (x, y)
        circle = plt.Circle((x, y), 0.3, color=colors[l], ec='black', linewidth=2, zorder=5)
        ax.add_patch(circle)
        if l == 0:
            ax.text(x, y, f'$x_{n+1}$', ha='center', va='center', fontsize=12, fontweight='bold', zorder=6)
        elif l == 1:
            ax.text(x, y, f'$h_{n+1}$', ha='center', va='center', fontsize=12, fontweight='bold', zorder=6)
        else:
            ax.text(x, y, '$y$', ha='center', va='center', fontsize=12, fontweight='bold', zorder=6)

# Connections
for l in range(len(layer_sizes) - 1):
    for n1 in range(layer_sizes[l]):
        for n2 in range(layer_sizes[l+1]):
            x1, y1 = positions[(l, n1)]
            x2, y2 = positions[(l+1, n2)]
            ax.plot([x1+0.3, x2-0.3], [y1, y2], 'k-', alpha=0.3, linewidth=1)

for l, name in enumerate(layer_names):
    ax.text(l*3, -max(layer_sizes)/2 - 0.8, name, ha='center', fontsize=11, fontweight='bold')

ax.set_xlim(-1, 7); ax.set_ylim(-3, 2.5)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Multi-Layer Perceptron (MLP) Architecture', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 2.2 MLP로 XOR 문제 해결

XOR은 다음과 같이 분해할 수 있습니다:

$$\text{XOR}(x_1, x_2) = \text{AND}(\text{OR}(x_1, x_2), \text{NAND}(x_1, x_2))$$

즉, 2개의 SLP(OR, NAND)를 결합하면 XOR을 해결할 수 있습니다!

### 수동 가중치 예제

- **은닉 뉴런 1** ($h_1$): $h_1 = \text{step}(x_1 + x_2 - 0.5)$ → OR와 유사
- **은닉 뉴런 2** ($h_2$): $h_2 = \text{step}(-x_1 - x_2 + 1.5)$ → NAND와 유사
- **출력 뉴런**: $y = \text{step}(h_1 + h_2 - 1.5)$ → AND of $h_1$, $h_2$

In [ ]:
# numpy로 수동 가중치 MLP XOR 해결
def step(z):
    return 1 if z >= 0 else 0

def mlp_xor_manual(x1, x2):
    # Hidden layer
    h1 = step(x1 + x2 - 0.5)    # OR-like
    h2 = step(-x1 - x2 + 1.5)   # NAND-like
    # Output layer
    y = step(h1 + h2 - 1.5)     # AND of h1, h2
    return h1, h2, y

print("=== Manual MLP for XOR ===")
print(f"{'x1':>3} {'x2':>3} │ {'h1 (OR)':>8} {'h2 (NAND)':>10} │ {'y (XOR)':>8}")
print("─" * 45)
for x1 in [0, 1]:
    for x2 in [0, 1]:
        h1, h2, y = mlp_xor_manual(x1, x2)
        expected = x1 ^ x2
        check = "✓" if y == expected else "✗"
        print(f"{x1:>3} {x2:>3} │ {h1:>8} {h2:>10} │ {y:>8} {check}")

### 🔗 AI 연결: 비선형 활성화 함수의 필수성

> 활성화 함수가 없거나 선형(identity)이면, 여러 층을 쌓아도 결국 **하나의 선형 변환**과 동일합니다:
>
> $$\mathbf{W}_2(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = \underbrace{(\mathbf{W}_2 \mathbf{W}_1)}_{\mathbf{W}'} \mathbf{x} + \underbrace{(\mathbf{W}_2 \mathbf{b}_1 + \mathbf{b}_2)}_{\mathbf{b}'}$$
>
> **비선형 활성화 함수가 있어야** 복잡한 패턴(비선형 결정 경계)을 학습할 수 있습니다!

### 📝 실습 2-1 [초급] MLP 순전파 직접 계산

주어진 가중치와 바이어스로 2-layer MLP의 순전파를 단계별로 직접 계산하세요.

**네트워크:** 입력 2 → 은닉 2 (ReLU) → 출력 1 (Sigmoid)

In [ ]:
# 📝 실습 2-1: MLP 순전파 직접 계산

# 주어진 파라미터
W1 = np.array([[0.5, -0.3],
               [0.2,  0.8]])  # (2, 2)
b1 = np.array([[0.1], [-0.2]])  # (2, 1)
W2 = np.array([[0.7, -0.5]])    # (1, 2)
b2 = np.array([[-0.1]])          # (1, 1)

# 입력
x = np.array([[1.0], [0.5]])  # (2, 1)

# === 순전파 ===
# Step 1: 은닉층 pre-activation
z1 = ____  # W1 @ x + b1

# Step 2: 은닉층 activation (ReLU)
h = ____  # ReLU(z1)

# Step 3: 출력층 pre-activation
z2 = ____  # W2 @ h + b2

# Step 4: 출력층 activation (Sigmoid)
y_hat = ____  # Sigmoid(z2)

print("=== MLP Forward Pass ===")
print(f"z1 = {z1.flatten()}")
print(f"h  = {h.flatten()}")
print(f"z2 = {z2.flatten()}")
print(f"y_hat = {y_hat.item():.6f}")

# 검증
assert np.allclose(z1, np.array([[0.45], [0.40]])), f"z1 incorrect: {z1.flatten()}"
assert np.allclose(h, np.array([[0.45], [0.40]])), f"h incorrect: {h.flatten()}"
assert np.allclose(z2, np.array([[0.015]]), atol=1e-6), f"z2 incorrect: {z2.flatten()}"
expected_yhat = 1 / (1 + np.exp(-0.015))
assert np.allclose(y_hat, expected_yhat, atol=1e-5), f"y_hat incorrect: {y_hat.item()}"

print("\n✅ 모든 테스트 통과!")

## 2.3 활성화 함수 심화 (Activation Functions)

| 함수 | 수식 | 출력 범위 | 미분 | 장점 | 단점 |
|:---:|:---:|:---:|:---:|:---:|:---:|
| **Sigmoid** | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $(0, 1)$ | $\sigma(z)(1-\sigma(z))$ | 확률 해석 가능 | Vanishing Gradient |
| **Tanh** | $\tanh(z)$ | $(-1, 1)$ | $1 - \tanh^2(z)$ | 0 중심 출력 | Vanishing Gradient |
| **ReLU** | $\max(0, z)$ | $[0, \infty)$ | $0$ or $1$ | 빠른 학습, 간단 | Dead Neuron 문제 |

### Vanishing Gradient 문제

Sigmoid/Tanh의 미분값은 최대 0.25/1.0입니다. 역전파 시 이 미분값들이 곱해지면서 gradient가 **지수적으로 줄어듭니다** (Vanishing Gradient). ReLU는 양수 영역에서 미분값이 항상 1이므로 이 문제를 완화합니다.

In [ ]:
# 활성화 함수 시각화
z = np.linspace(-5, 5, 200)

# Functions
sigmoid_vals = 1 / (1 + np.exp(-z))
tanh_vals = np.tanh(z)
relu_vals = np.maximum(0, z)

# Derivatives
sigmoid_d = sigmoid_vals * (1 - sigmoid_vals)
tanh_d = 1 - tanh_vals**2
relu_d = (z > 0).astype(float)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

funcs = [sigmoid_vals, tanh_vals, relu_vals]
derivs = [sigmoid_d, tanh_d, relu_d]
names = ['Sigmoid', 'Tanh', 'ReLU']
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i in range(3):
    # Function
    axes[0, i].plot(z, funcs[i], color=colors[i], linewidth=2)
    axes[0, i].axhline(y=0, color='k', linewidth=0.5)
    axes[0, i].axvline(x=0, color='k', linewidth=0.5)
    axes[0, i].set_title(f'{names[i]}', fontsize=14, fontweight='bold')
    axes[0, i].set_xlabel('z'); axes[0, i].set_ylabel('f(z)')
    axes[0, i].grid(True, alpha=0.3)

    # Derivative
    axes[1, i].plot(z, derivs[i], color=colors[i], linewidth=2, linestyle='--')
    axes[1, i].axhline(y=0, color='k', linewidth=0.5)
    axes[1, i].axvline(x=0, color='k', linewidth=0.5)
    axes[1, i].set_title(f'{names[i]} Derivative', fontsize=14, fontweight='bold')
    axes[1, i].set_xlabel('z'); axes[1, i].set_ylabel("f'(z)")
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Activation Functions and Their Derivatives', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### 🔗 AI 연결: Vanishing Gradient → ReLU의 등장

Sigmoid를 여러 층 통과시키면 gradient가 어떻게 변하는지 확인해봅시다.

In [ ]:
# Vanishing Gradient 시연
def sigmoid_fn(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_grad(z):
    s = sigmoid_fn(z)
    return s * (1 - s)

# 10층 네트워크에서 gradient 전파 시뮬레이션
n_layers = 10
initial_gradient = 1.0

# Sigmoid chain
grad_sigmoid = [initial_gradient]
for _ in range(n_layers):
    grad_sigmoid.append(grad_sigmoid[-1] * sigmoid_grad(0.5))

# ReLU chain (z > 0 가정)
grad_relu = [initial_gradient]
for _ in range(n_layers):
    grad_relu.append(grad_relu[-1] * 1.0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(n_layers+1), grad_sigmoid, 'r-o', label='Sigmoid', linewidth=2, markersize=8)
ax.plot(range(n_layers+1), grad_relu, 'g-s', label='ReLU', linewidth=2, markersize=8)
ax.set_xlabel('Layer (from output to input)', fontsize=12)
ax.set_ylabel('Gradient Magnitude', fontsize=12)
ax.set_title('Vanishing Gradient: Sigmoid vs ReLU', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout(); plt.show()

print(f"Sigmoid: After {n_layers} layers, gradient = {grad_sigmoid[-1]:.2e}")
print(f"ReLU:    After {n_layers} layers, gradient = {grad_relu[-1]:.2e}")

### 📝 실습 2-2 [중급] 활성화 함수 구현 + 미분 비교

`numpy`로 sigmoid, tanh, relu와 그 미분을 직접 구현하고, 여러 층 통과 후 gradient 크기를 비교하세요.

In [ ]:
# 📝 실습 2-2: 활성화 함수 직접 구현

def my_sigmoid(z):
    ____  # 1 / (1 + exp(-z))

def my_sigmoid_deriv(z):
    ____  # sigma(z) * (1 - sigma(z))

def my_tanh(z):
    ____  # np.tanh(z)

def my_tanh_deriv(z):
    ____  # 1 - tanh(z)^2

def my_relu(z):
    ____  # max(0, z)

def my_relu_deriv(z):
    ____  # 0 or 1

# 검증
test_z = np.array([-1.0, 0.0, 1.0, 2.0])

# Sigmoid 검증
assert np.allclose(my_sigmoid(test_z), 1/(1+np.exp(-test_z))), "Sigmoid 구현 오류"
assert np.allclose(my_sigmoid_deriv(test_z), my_sigmoid(test_z)*(1-my_sigmoid(test_z))), "Sigmoid 미분 오류"

# Tanh 검증
assert np.allclose(my_tanh(test_z), np.tanh(test_z)), "Tanh 구현 오류"
assert np.allclose(my_tanh_deriv(test_z), 1-np.tanh(test_z)**2), "Tanh 미분 오류"

# ReLU 검증
assert np.allclose(my_relu(test_z), np.maximum(0, test_z)), "ReLU 구현 오류"

print("\n✅ 모든 테스트 통과!")

# Gradient 크기 비교 시각화
n_layers_ex = 15
z_val = 0.5
funcs_grad = {
    'Sigmoid': lambda: my_sigmoid_deriv(z_val),
    'Tanh': lambda: my_tanh_deriv(z_val),
    'ReLU': lambda: 1.0  # z > 0
}

fig, ax = plt.subplots(figsize=(10, 5))
for name, grad_fn in funcs_grad.items():
    grads = [1.0]
    for _ in range(n_layers_ex):
        grads.append(grads[-1] * grad_fn())
    ax.plot(range(n_layers_ex+1), grads, '-o', label=name, linewidth=2, markersize=6)

ax.set_xlabel('Layer (from output to input)', fontsize=12)
ax.set_ylabel('Gradient Magnitude', fontsize=12)
ax.set_title('Gradient Propagation Through Layers', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout(); plt.show()

## 2.4 PyTorch로 MLP 구축

### 핵심 구성요소

| 구성요소 | PyTorch API | 설명 |
|:---:|:---:|:---|
| 모델 | `nn.Module` | 신경망 클래스 정의 |
| 층 | `nn.Linear(in, out)` | 완전 연결층 (Fully Connected) |
| 활성화 | `nn.ReLU()`, `nn.Sigmoid()` | 비선형 활성화 함수 |
| 손실함수 | `nn.MSELoss()`, `nn.BCELoss()`, `nn.CrossEntropyLoss()` | 예측-정답 차이 |
| 옵티마이저 | `optim.SGD()`, `optim.Adam()` | 파라미터 업데이트 |

In [ ]:
# PyTorch MLP로 XOR 학습
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        h = self.relu(self.layer1(x))
        return self.sigmoid(self.layer2(h))

# XOR data
X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

torch.manual_seed(42)
model = MLP(2, 8, 1)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

losses = []
for epoch in range(1000):
    output = model(X)
    loss = criterion(output, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# Results
with torch.no_grad():
    predictions = model(X)
    predicted = (predictions > 0.5).float()
    print("XOR Results:")
    for i in range(4):
        print(f"  Input: {X[i].tolist()} → Pred: {predictions[i].item():.4f} → Class: {int(predicted[i].item())}")

# Plot loss
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('XOR Training Loss', fontweight='bold')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### 📝 실습 2-3 [중급] PyTorch MLP로 XOR 학습

직접 MLP 클래스를 정의하고 학습 루프를 작성하세요.

In [ ]:
# 📝 실습 2-3: PyTorch MLP로 XOR

class MyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        ____  # nn.Linear(2, 8)
        ____  # nn.Linear(8, 1)
        ____  # nn.ReLU()
        ____  # nn.Sigmoid()

    def forward(self, x):
        x = ____  # layer1 → relu → layer2 → sigmoid
        return x

# 데이터
X_xor_t = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y_xor_t = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

torch.manual_seed(0)
my_model = MyMLP()
criterion_ex = nn.BCELoss()
optimizer_ex = torch.optim.Adam(my_model.parameters(), lr=0.01)

# 학습
ex_losses = []
for epoch in range(2000):
    output = my_model(X_xor_t)
    loss = criterion_ex(output, y_xor_t)
    optimizer_ex.zero_grad()
    ____  # loss.backward()
    ____  # optimizer.step()
    ex_losses.append(loss.item())

# 결정 경계 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(ex_losses)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('XOR Training Loss', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid_t = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    Z = my_model(grid_t).numpy().reshape(xx.shape)

axes[1].contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.6)
axes[1].contour(xx, yy, Z, levels=[0.5], colors='green', linewidths=2)
xor_colors = ['blue', 'red', 'red', 'blue']
for i in range(4):
    axes[1].scatter(X_xor_t[i,0], X_xor_t[i,1], c=xor_colors[i], s=200, edgecolors='k', zorder=5)
axes[1].set_xlabel('x1'); axes[1].set_ylabel('x2')
axes[1].set_title('XOR Decision Boundary (MLP)', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# 검증
with torch.no_grad():
    final_pred = (my_model(X_xor_t) > 0.5).float()
    xor_acc = (final_pred == y_xor_t).float().mean().item()
    print(f"XOR Accuracy: {xor_acc:.4f}")

assert xor_acc == 1.0, f"XOR accuracy {xor_acc} != 1.0. 더 학습하거나 구조를 수정하세요!"
print("\n✅ 모든 테스트 통과!")

### 🏆 도전 문제 [고급] 은닉층 크기에 따른 결정경계 변화

은닉층의 뉴런 수를 바꾸면 결정 경계가 어떻게 변하는지 관찰하세요.

In [ ]:
# 🏆 도전: 은닉층 크기별 결정경계

hidden_sizes = [1, 2, 4, 8, 16, 32]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

X_ch = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y_ch = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

for idx, h_size in enumerate(hidden_sizes):
    class FlexMLP(nn.Module):
        def __init__(self):
            super().__init__()
            ____  # nn.Linear(2, h_size)
            ____  # nn.Linear(h_size, 1)

        def forward(self, x):
            ____  # fc1 → relu → fc2 → sigmoid

    torch.manual_seed(42)
    flex_model = FlexMLP()
    opt_ch = torch.optim.Adam(flex_model.parameters(), lr=0.01)
    crit_ch = nn.BCELoss()

    for epoch in range(3000):
        out = flex_model(X_ch)
        loss = crit_ch(out, y_ch)
        opt_ch.zero_grad()
        loss.backward()
        opt_ch.step()

    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
    grid_ch = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        Z = flex_model(grid_ch).numpy().reshape(xx.shape)
        acc = ((flex_model(X_ch) > 0.5).float() == y_ch).float().mean().item()

    axes[idx].contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.6)
    axes[idx].contour(xx, yy, Z, levels=[0.5], colors='green', linewidths=2)
    xor_c = ['blue', 'red', 'red', 'blue']
    for i in range(4):
        axes[idx].scatter(X_ch[i,0], X_ch[i,1], c=xor_c[i], s=150, edgecolors='k', zorder=5)
    axes[idx].set_title(f'Hidden={h_size}, Acc={acc:.2f}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('x1'); axes[idx].set_ylabel('x2')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('XOR Decision Boundary vs Hidden Layer Size', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()

---
# Part 3: 역전파 (Backpropagation)

## 3.1 순전파 복습 (Forward Pass Review)

2-layer 네트워크의 순전파 과정을 단계별로 추적합니다:

1. **입력**: $\mathbf{x}$
2. **1층 pre-activation**: $\mathbf{z}_1 = \mathbf{W}_1 \mathbf{x} + \mathbf{b}_1$
3. **1층 activation**: $\mathbf{h} = \sigma(\mathbf{z}_1)$
4. **2층 pre-activation**: $z_2 = \mathbf{W}_2 \mathbf{h} + b_2$
5. **2층 activation (출력)**: $\hat{y} = \sigma(z_2)$
6. **손실**: $L = (y - \hat{y})^2$

In [ ]:
# 순전파 값 추적
import numpy as np

def sigmoid_fwd(z):
    return 1 / (1 + np.exp(-z))

np.random.seed(42)
W1_bp = np.random.randn(2, 2) * 0.5
b1_bp = np.zeros((2, 1))
W2_bp = np.random.randn(1, 2) * 0.5
b2_bp = np.zeros((1, 1))

x_bp = np.array([[1], [0]])  # input
y_bp = np.array([[1]])        # target

# Forward pass with value tracking
z1_bp = W1_bp @ x_bp + b1_bp
h_bp = sigmoid_fwd(z1_bp)
z2_bp = W2_bp @ h_bp + b2_bp
yhat_bp = sigmoid_fwd(z2_bp)
loss_bp = (y_bp - yhat_bp) ** 2

print("=== Forward Pass (Value Tracking) ===")
print(f"x     = {x_bp.flatten()}")
print(f"W1    = \n{W1_bp}")
print(f"b1    = {b1_bp.flatten()}")
print(f"z1    = {z1_bp.flatten()}")
print(f"h     = {h_bp.flatten()}")
print(f"W2    = {W2_bp.flatten()}")
print(f"b2    = {b2_bp.flatten()}")
print(f"z2    = {z2_bp.flatten()}")
print(f"y_hat = {yhat_bp.item():.6f}")
print(f"Loss  = {loss_bp.item():.6f}")

## 3.2 연쇄법칙 복습 (Chain Rule)

### 단일 변수

$$\frac{df}{dx} = \frac{df}{du} \cdot \frac{du}{dx}$$

### 다변수 (Computational Graph)

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial z} \cdot \frac{\partial z}{\partial w}$$

PyTorch의 `autograd`는 이 연쇄법칙을 **자동으로** 적용합니다!

In [ ]:
# torch.autograd vs 수동 계산
# f(x) = (x^2 + 3x)(2x - 1) at x=2
x_auto = torch.tensor(2.0, requires_grad=True)
u = x_auto**2 + 3*x_auto
v = 2*x_auto - 1
f = u * v
f.backward()
print(f"Autograd: df/dx = {x_auto.grad.item()}")

# Manual: df/dx = du/dx * v + u * dv/dx = (2x+3)(2x-1) + (x^2+3x)(2)
# At x=2: = (7)(3) + (10)(2) = 21 + 20 = 41
manual_result = (2*2+3)*(2*2-1) + (2**2+3*2)*2
print(f"Manual:   df/dx = {manual_result}")

## 3.3 역전파 알고리즘 상세 (Backpropagation)

### 네트워크 구조
입력 $\mathbf{x}$ → 은닉 $\mathbf{h}$ → 출력 $\hat{y}$ → 손실 $L$

### Forward Pass
1. $\mathbf{z}_1 = \mathbf{W}_1 \mathbf{x} + \mathbf{b}_1$
2. $\mathbf{h} = \sigma(\mathbf{z}_1)$
3. $z_2 = \mathbf{W}_2 \mathbf{h} + b_2$
4. $\hat{y} = \sigma(z_2)$
5. $L = (y - \hat{y})^2$

### Backward Pass (연쇄법칙 적용)

출력층에서 입력층 방향으로 gradient를 전파합니다:

$$\frac{\partial L}{\partial \hat{y}} = -2(y - \hat{y})$$

$$\frac{\partial L}{\partial z_2} = \frac{\partial L}{\partial \hat{y}} \cdot \sigma'(z_2)$$

$$\frac{\partial L}{\partial \mathbf{W}_2} = \frac{\partial L}{\partial z_2} \cdot \mathbf{h}^T$$

$$\frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial z_2}$$

$$\frac{\partial L}{\partial \mathbf{h}} = \mathbf{W}_2^T \cdot \frac{\partial L}{\partial z_2}$$

$$\frac{\partial L}{\partial \mathbf{z}_1} = \frac{\partial L}{\partial \mathbf{h}} \cdot \sigma'(\mathbf{z}_1)$$

$$\frac{\partial L}{\partial \mathbf{W}_1} = \frac{\partial L}{\partial \mathbf{z}_1} \cdot \mathbf{x}^T$$

$$\frac{\partial L}{\partial \mathbf{b}_1} = \frac{\partial L}{\partial \mathbf{z}_1}$$

In [ ]:
# 완전한 역전파 구현 (numpy)
import numpy as np

def sigmoid_bp(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv_bp(z):
    s = sigmoid_bp(z)
    return s * (1 - s)

# Network: 2 inputs → 2 hidden → 1 output
np.random.seed(42)
W1 = np.random.randn(2, 2) * 0.5
b1 = np.zeros((2, 1))
W2 = np.random.randn(1, 2) * 0.5
b2 = np.zeros((1, 1))

# Single training example: XOR (1,0) → 1
x = np.array([[1], [0]])  # (2,1)
y = np.array([[1]])        # (1,1)

# === Forward Pass ===
z1 = W1 @ x + b1           # (2,1)
h = sigmoid_bp(z1)          # (2,1)
z2 = W2 @ h + b2            # (1,1)
y_hat = sigmoid_bp(z2)      # (1,1)
loss = (y - y_hat) ** 2     # scalar

print("=== Forward Pass ===")
print(f"z1 = {z1.flatten()}")
print(f"h  = {h.flatten()}")
print(f"z2 = {z2.flatten()}")
print(f"y_hat = {y_hat.item():.6f}")
print(f"L  = {loss.item():.6f}")

# === Backward Pass ===
dL_dy_hat = -2 * (y - y_hat)           # dL/dy_hat
dL_dz2 = dL_dy_hat * sigmoid_deriv_bp(z2)  # dL/dz2
dL_dW2 = dL_dz2 @ h.T                  # dL/dW2
dL_db2 = dL_dz2                         # dL/db2
dL_dh = W2.T @ dL_dz2                  # dL/dh
dL_dz1 = dL_dh * sigmoid_deriv_bp(z1)  # dL/dz1
dL_dW1 = dL_dz1 @ x.T                  # dL/dW1
dL_db1 = dL_dz1                         # dL/db1

print("\n=== Backward Pass (Gradients) ===")
print(f"dL/dW2 = {dL_dW2.flatten()}")
print(f"dL/db2 = {dL_db2.flatten()}")
print(f"dL/dW1 = {dL_dW1.flatten()}")
print(f"dL/db1 = {dL_db1.flatten()}")

In [ ]:
# PyTorch autograd로 검증
W1_t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1_t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2_t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2_t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)
x_t = torch.tensor(x, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)

z1_t = W1_t @ x_t + b1_t
h_t = torch.sigmoid(z1_t)
z2_t = W2_t @ h_t + b2_t
y_hat_t = torch.sigmoid(z2_t)
loss_t = (y_t - y_hat_t) ** 2
loss_t.backward()

print("=== PyTorch Autograd Verification ===")
print(f"dL/dW2 (torch) = {W2_t.grad.flatten().tolist()}")
print(f"dL/dW2 (numpy) = {dL_dW2.flatten().tolist()}")
print(f"Match: {np.allclose(W2_t.grad.numpy(), dL_dW2)}")
print(f"dL/dW1 (torch) = {W1_t.grad.flatten().tolist()}")
print(f"dL/dW1 (numpy) = {dL_dW1.flatten().tolist()}")
print(f"Match: {np.allclose(W1_t.grad.numpy(), dL_dW1)}")

### 🔗 AI 연결: PyTorch autograd = 자동 역전파

> PyTorch의 `autograd`는 순전파 과정에서 **computational graph**를 자동으로 구성하고, `.backward()` 호출 시 연쇄법칙을 적용하여 모든 gradient를 자동으로 계산합니다. 위에서 numpy로 직접 구현한 역전파와 PyTorch autograd의 결과가 정확히 일치하는 것을 확인했습니다!

### 📝 실습 3-1 [초급] BP 수식 추적

주어진 값으로 forward + backward pass를 수동으로 계산하세요.

In [ ]:
# 📝 실습 3-1: BP 수식 추적

# 주어진 파라미터
np.random.seed(100)
W1_ex = np.array([[0.3, 0.7], [-0.5, 0.2]])
b1_ex = np.array([[0.1], [-0.3]])
W2_ex = np.array([[0.6, -0.4]])
b2_ex = np.array([[0.2]])
x_ex = np.array([[0.5], [0.8]])
y_ex = np.array([[1.0]])

# Forward pass
z1_ex = ____  # W1_ex @ x_ex + b1_ex
h_ex = ____  # sigmoid(z1_ex)
z2_ex = ____  # W2_ex @ h_ex + b2_ex
yhat_ex = ____  # sigmoid(z2_ex)
loss_ex = (y_ex - yhat_ex) ** 2

print("=== Forward ===")
print(f"z1    = {z1_ex.flatten()}")
print(f"h     = {h_ex.flatten()}")
print(f"z2    = {z2_ex.item():.6f}")
print(f"y_hat = {yhat_ex.item():.6f}")
print(f"Loss  = {loss_ex.item():.6f}")

# Backward pass
dL_dyhat_ex = -2 * (y_ex - yhat_ex)
dLdz2_ex = ____  # dL/dy_hat * sigmoid'(z2)
dLdW2_ex = ____  # dL/dz2 @ h^T
dLdb2_ex = dLdz2_ex
dLdh_ex = W2_ex.T @ dLdz2_ex
dLdz1_ex = dLdh_ex * (h_ex * (1 - h_ex))
dLdW1_ex = ____  # dL/dz1 @ x^T
dLdb1_ex = dLdz1_ex

print("\n=== Backward ===")
print(f"dL/dW2 = {dLdW2_ex.flatten()}")
print(f"dL/dW1 = {dLdW1_ex.flatten()}")

# PyTorch 검증
W1_v = torch.tensor(W1_ex, dtype=torch.float64, requires_grad=True)
b1_v = torch.tensor(b1_ex, dtype=torch.float64, requires_grad=True)
W2_v = torch.tensor(W2_ex, dtype=torch.float64, requires_grad=True)
b2_v = torch.tensor(b2_ex, dtype=torch.float64, requires_grad=True)
x_v = torch.tensor(x_ex, dtype=torch.float64)
y_v = torch.tensor(y_ex, dtype=torch.float64)

z1v = W1_v @ x_v + b1_v
hv = torch.sigmoid(z1v)
z2v = W2_v @ hv + b2_v
yhv = torch.sigmoid(z2v)
lv = (y_v - yhv) ** 2
lv.backward()

assert np.allclose(dLdW2_ex, W2_v.grad.numpy(), atol=1e-7), "dL/dW2 mismatch"
assert np.allclose(dLdW1_ex, W1_v.grad.numpy(), atol=1e-7), "dL/dW1 mismatch"
assert np.allclose(dLdb2_ex, b2_v.grad.numpy(), atol=1e-7), "dL/db2 mismatch"
assert np.allclose(dLdb1_ex, b1_v.grad.numpy(), atol=1e-7), "dL/db1 mismatch"

print("\n✅ 모든 테스트 통과!")

### 📝 실습 3-2 [중급] numpy로 완전한 BP 학습 루프 구현

XOR 4개 데이터에 대해 역전파로 학습하는 완전한 코드를 작성하세요.

In [ ]:
# 📝 실습 3-2: numpy BP 학습 루프

# XOR 데이터
X_bp_data = [np.array([[0],[0]]), np.array([[0],[1]]), np.array([[1],[0]]), np.array([[1],[1]])]
y_bp_data = [np.array([[0]]), np.array([[1]]), np.array([[1]]), np.array([[0]])]

# 초기 가중치
np.random.seed(1)
W1_np = np.random.randn(4, 2) * 0.5
b1_np = np.zeros((4, 1))
W2_np = np.random.randn(1, 4) * 0.5
b2_np = np.zeros((1, 1))

lr = 1.0
losses_bp = []

for epoch in range(1000):
    epoch_loss = 0
    gW1 = np.zeros_like(W1_np)
    gb1 = np.zeros_like(b1_np)
    gW2 = np.zeros_like(W2_np)
    gb2 = np.zeros_like(b2_np)

    for xi, yi in zip(X_bp_data, y_bp_data):
        # Forward
        z1_np = ____  # W1_np @ xi + b1_np
        h_np = ____  # sigmoid(z1_np)
        z2_np = ____  # W2_np @ h_np + b2_np
        yh = ____  # sigmoid(z2_np)
        epoch_loss += (yi - yh) ** 2

        # Backward
        dL_dyh = -2 * (yi - yh)
        dL_dz2_np = ____  # dL/dy_hat * sigmoid'(z2)
        gW2 += ____  # dL/dz2 @ h^T
        gb2 += dL_dz2_np
        dL_dz1_np = ____  # (W2^T @ dL/dz2) * sigmoid'(z1)
        gW1 += ____  # dL/dz1 @ x^T
        gb1 += dL_dz1_np

    # Average gradients
    gW1 /= 4; gb1 /= 4; gW2 /= 4; gb2 /= 4

    # Update weights
    ____  # W1 -= lr * gW1, 나머지도 동일하게 업데이트

    losses_bp.append(epoch_loss.item() / 4)

# 결과 확인
print("=== XOR BP Training Results ===")
correct_bp = 0
for xi, yi in zip(X_bp_data, y_bp_data):
    z1_np = W1_np @ xi + b1_np
    h_np = 1 / (1 + np.exp(-z1_np))
    z2_np = W2_np @ h_np + b2_np
    yh = 1 / (1 + np.exp(-z2_np))
    pred = 1 if yh > 0.5 else 0
    correct_bp += (pred == yi.item())
    print(f"  Input: {xi.flatten()} → Pred: {yh.item():.4f} → Class: {pred} (Target: {int(yi.item())})")

acc_bp = correct_bp / 4
print(f"\nAccuracy: {acc_bp:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses_bp)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('XOR Backpropagation Training (NumPy)', fontweight='bold')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

assert acc_bp == 1.0, f"Accuracy {acc_bp} != 1.0"
print("\n✅ 모든 테스트 통과!")

## 3.4 학습 루프 (Training Loop)

신경망의 학습은 다음 4단계를 반복합니다:

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  Forward     │ →   │  Loss        │ →   │  Backward    │ →   │  Update      │
│  Pass        │     │  Computation │     │  Pass (BP)   │     │  (GD)        │
│  y = f(x)   │     │  L = loss(y) │     │  dL/dW      │     │  W -= a*dL/dW│
└─────────────┘     └─────────────┘     └─────────────┘     └─────────────┘
```

PyTorch에서의 학습 루프 패턴:
```python
for epoch in range(n_epochs):
    output = model(X)           # Forward
    loss = criterion(output, y) # Loss
    optimizer.zero_grad()       # Gradient 초기화
    loss.backward()             # Backward (BP)
    optimizer.step()            # Update (GD)
```

### 📝 실습 3-3 [중급] PyTorch 학습 루프 작성

간단한 회귀 데이터 ($y = 2x + 1 + \text{noise}$)를 MLP로 학습하세요.

In [ ]:
# 📝 실습 3-3: PyTorch 학습 루프 (회귀)

# 데이터 생성: y = 2x + 1 + noise
torch.manual_seed(42)
X_reg = torch.linspace(-3, 3, 100).unsqueeze(1)
y_reg = 2 * X_reg + 1 + torch.randn_like(X_reg) * 0.3

# 모델 정의
reg_model = ____  # nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16,1))

mse_loss = nn.MSELoss()
opt_reg = torch.optim.Adam(reg_model.parameters(), lr=0.01)

# 학습
reg_losses = []
for epoch in range(500):
    outputs_reg = ____  # reg_model(X_reg)
    loss_reg = ____  # MSE Loss 계산
    opt_reg.zero_grad()
    ____  # loss_reg.backward()
    ____  # opt_reg.step()
    reg_losses.append(loss_reg.item())

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(reg_losses, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Regression Training Loss', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Predictions
with torch.no_grad():
    y_pred_reg = reg_model(X_reg)
axes[1].scatter(X_reg.numpy(), y_reg.numpy(), c='blue', alpha=0.3, s=20, label='Data')
axes[1].plot(X_reg.numpy(), y_pred_reg.numpy(), 'r-', linewidth=2, label='Model Prediction')
axes[1].plot(X_reg.numpy(), (2*X_reg+1).numpy(), 'g--', linewidth=2, label='True: y=2x+1')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
axes[1].set_title('Regression Result', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# 검증: 최종 loss가 충분히 작은지
assert reg_losses[-1] < 0.5, f"Final loss {reg_losses[-1]:.4f} too high!"
print("\n✅ 모든 테스트 통과!")

---
# Part 4: 종합 실습 — 직접 만드는 신경망

## 🏆 종합 프로젝트 [고급] MNIST 손글씨 분류

이 실습에서는 앞에서 배운 모든 개념을 종합하여 MNIST 손글씨 숫자(0-9) 분류 모델을 직접 구현합니다.

**미션:** Test Accuracy 95% 이상 달성

### 가이드라인

**Step 1: 데이터 로드**
- `torchvision.datasets.MNIST` 사용
- `transforms.ToTensor()`로 텐서 변환
- `DataLoader`로 배치 구성 (`batch_size=64`)

**Step 2: 데이터 탐색**
- 이미지 shape 확인 (28x28 = 784 features)
- 샘플 이미지 5개 시각화

**Step 3: 디바이스 설정**
- 위에서 설정한 `device` 사용
- `model.to(device)`, `data.to(device)`

**Step 4: MLP 모델 정의**
- `nn.Module` 상속
- 784 → 256 → 128 → 10 구조 권장
- 활성화 함수: ReLU (은닉층)
- 출력층: 활성화 함수 없음 (`CrossEntropyLoss`가 내부적으로 softmax 적용)

**Step 5: 손실함수 + 옵티마이저**
- `nn.CrossEntropyLoss()`
- `torch.optim.Adam(model.parameters(), lr=0.001)`

**Step 6: 학습 루프**
- 10 epochs
- 매 epoch: train loss 출력
- 구조: `for batch in dataloader → forward → loss → backward → step`

**Step 7: 평가 + 시각화**
- Test accuracy 계산
- Loss curve 그리기
- 예측 결과 샘플 시각화 (정답/오답 구분)

In [ ]:
# 🏆 종합 프로젝트: MNIST 손글씨 분류

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# === Step 1: 데이터 로드 ===
transform = transforms.ToTensor()

train_dataset = ____  # torchvision.datasets.MNIST(...)
test_dataset = ____  # torchvision.datasets.MNIST(...)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# === Step 2: 데이터 탐색 ===
images_sample, labels_sample = next(iter(train_loader))
print(f"Batch shape: {images_sample.shape}")  # [64, 1, 28, 28]
print(f"Labels shape: {labels_sample.shape}")  # [64]

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    axes[i].imshow(images_sample[i].squeeze(), cmap='gray')
    axes[i].set_title(f'Label: {labels_sample[i].item()}', fontsize=12)
    axes[i].axis('off')
plt.suptitle('MNIST Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# === Step 3: 모델 정의 ===
class MNISTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        ____  # 784 → 256 → 128 → 10 MLP를 정의하세요

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten: [B, 1, 28, 28] → [B, 784]
        ____  # fc1 → relu → fc2 → relu → fc3
        return x

model_mn = MNISTClassifier().to(device)
print(f"\nModel: {model_mn}")
total_params = sum(p.numel() for p in model_mn.parameters())
print(f"Total parameters: {total_params:,}")

# === Step 4: 손실함수 + 옵티마이저 ===
criterion_mn = ____  # CrossEntropyLoss
optimizer_mn = ____  # Adam, lr=0.001

# === Step 5: 학습 ===
n_epochs = 10
train_losses = []

for epoch in range(n_epochs):
    model_mn.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        ____  # Forward → Loss → Backward → Update 를 작성하세요

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{n_epochs}], Loss: {avg_loss:.4f}")

# === Step 6: 평가 ===
model_mn.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_mn(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f"\nTest Accuracy: {accuracy:.4f} ({correct}/{total})")
assert accuracy > 0.95, f"Accuracy {accuracy:.4f} < 0.95. 모델을 개선해보세요!"
print("\n✅ 모든 테스트 통과! Test Accuracy > 95% 달성!")

# === Step 7: 시각화 ===
# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(train_losses, 'b-', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('MNIST Training Loss', fontweight='bold')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---

## 📋 핵심 개념 정리 (Summary)

| 개념 | 핵심 내용 | 수식 |
|:---|:---|:---|
| **Perceptron** | 입력의 가중합 + 활성화 | $y = f(\mathbf{w} \cdot \mathbf{x} + b)$ |
| **SLP 한계** | 선형 분리 가능한 문제만 해결 | XOR 불가 |
| **MLP** | 은닉층 추가 → 비선형 문제 해결 | $\mathbf{h} = f(\mathbf{W}_1\mathbf{x} + \mathbf{b}_1)$ |
| **활성화 함수** | 비선형성 부여 | ReLU, Sigmoid, Tanh |
| **Forward Pass** | 입력 → 은닉 → 출력 순전파 | $\hat{y} = g(\mathbf{W}_2\mathbf{h} + \mathbf{b}_2)$ |
| **Loss Function** | 예측-정답 차이 정량화 | MSE, BCE, CE |
| **Backpropagation** | 연쇄법칙으로 gradient 역전파 | $\frac{\partial L}{\partial W} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial W}$ |
| **Gradient Descent** | gradient 방향으로 파라미터 업데이트 | $W \leftarrow W - \alpha \cdot \frac{\partial L}{\partial W}$ |

> **핵심:** SLP(단일 뉴런) → MLP(다층) → BP(학습) → GD(최적화) — 이 네 가지가 딥러닝의 기초!

---

## 🚀 더 알아보기 (Further Reading)

- **Understanding Deep Learning** (Prince) Ch 3-7
- **Dive into Deep Learning** (d2l.ai) Ch 5: MLP
- **3Blue1Brown** "Neural Networks" series (YouTube)

---

*End of Lab #05 — Artificial Neural Network*